<a href="https://colab.research.google.com/github/AnnesaAD05/Python-Projects-Informatics-1st-year-annesa/blob/main/%F0%9F%8C%A4%EF%B8%8F_Mood_Based_Weather_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🌤️ Mood-Based Weather Assistant


In [12]:
# Install necessary libraries
!pip install requests google-generativeai

import requests
from google.colab import userdata
import google.generativeai as genai

# --- FIX: Update deprecated library import ---
# The previous import was: import google.generativeai as genai
# The recommended new import is:
import google.generativeai as genai # Keep for now to avoid breaking existing code, but inform user

# Or, for new code, use:
# import google.genai as genai
# For simplicity and to avoid breaking existing references to 'genai', we'll keep the alias.
# However, it's good practice to update to 'google.genai' for new projects.



In [13]:
# Retrieve API keys from Colab secrets
OPENWEATHER_API_KEY = userdata.get('OPENWEATHER_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Configure Google Gemini API
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-pro')

print("API clients initialized. Make sure you have set 'OPENWEATHER_API_KEY' and 'GOOGLE_API_KEY' in Colab secrets.")

API clients initialized. Make sure you have set 'OPENWEATHER_API_KEY' and 'GOOGLE_API_KEY' in Colab secrets.


In [14]:
def get_weather_data(city, api_key):
    base_url = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        'q': city,
        'appid': api_key,
        'units': 'metric'  # For Celsius, use 'imperial' for Fahrenheit
    }
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        weather_data = response.json()
        return weather_data
    except requests.exceptions.RequestException as e:
        print(f"Error fetching weather data: {e}")
        return None

In [15]:
def generate_mood_suggestions(weather_description, temperature, city):
    prompt = f"""
Based on the following weather conditions in {city}:
- Weather: {weather_description}
- Temperature: {temperature}°C

Provide short, mood-based suggestions, including:
1.  An emoji describing the weather.
2.  A brief statement about the weather.
3.  One or two 'Suggested:' activities.
4.  One 'Outdoor activity:' suggestion with a brief comment.

Format your response as follows:
[Emoji] [City] — [Temperature]°C
Weather is [weather description].
🎧 Suggested: [Activity 1] + [Activity 2]
🚶 Outdoor activity: [Outdoor activity suggestion].

Example for 'Cloudy, 31°C' in Delhi:
☁️ Delhi — 31°C
Weather is cloudy today.
🎧 Suggested: Indoor study + café time
🚶 Outdoor activity: Possible, but carry an umbrella.

Now, generate suggestions for the given conditions:
"""
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"Error generating suggestions with Gemini: {e}")
        return """I'm sorry, I couldn't generate suggestions at this time. Please check your API key or try again later."""

In [16]:
def run_weather_assistant(city):
    weather_data = get_weather_data(city, OPENWEATHER_API_KEY)

    if weather_data:
        main_weather = weather_data['weather'][0]['main']
        description = weather_data['weather'][0]['description']
        temperature = weather_data['main']['temp']

        weather_summary = f"{main_weather}, {description}"
        print(f"Current weather in {city}: {weather_summary}, {temperature}°C\n")

        suggestions = generate_mood_suggestions(description, temperature, city)
        print(suggestions)
    else:
        print(f"Could not retrieve weather for {city}. Please check the city name and your API key.")

In [18]:
run_weather_assistant('Delhi')

Error fetching weather data: 401 Client Error: Unauthorized for url: http://api.openweathermap.org/data/2.5/weather?q=Delhi&appid=f290ba7e10d7f4884a2fa15e9cf223ca&units=metric
Could not retrieve weather for Delhi. Please check the city name and your API key.
